# # Deep Dive Feature Exploration
# 
# Goals:
# 1. Recursively scan all columns in the flattened dataframe.
# 2. Extract every potential signal (high variance, non-null).
# 3. Correlate with `fraud_flag` to find hidden gems.
# 4. Generate a report for `knowledge_base.md`.

In [ ]:
import polars as pl
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

DATA_PATH = "artifacts/raw_insertions.parquet"

print(f"Loading {DATA_PATH}...")
df = pl.read_parquet(DATA_PATH)

# ## 1. Feature Enumeration & Basic filtering

In [ ]:
# Derive Target
df = df.with_columns(
    pl.col("fraud_flag").is_not_null().alias("is_fraud").cast(pl.Int8)
)

all_columns = df.columns
total_rows = df.height
print(f"Total Columns: {len(all_columns)}")
print(f"Total Rows: {total_rows}")

# Filter out ID columns for feature analysis, but keep hashes if they might be effectively categorical
# We skip columns that are purely unique IDs (count distinct == count)
# But hashes like 'city_hash' are useful.

potential_features = []
stats = []

# Sampling for speed if needed
if total_rows > 100000:
    df_sample = df.sample(100000, seed=42)
else:
    df_sample = df

print("Scanning columns...")

for col in all_columns:
    if col in ["is_fraud", "fraud_flag", "listing_id"]:
        continue
        
    # Check Type
    dtype = df_sample.schema[col]
    
    # 1. Null Check
    null_count = df_sample[col].null_count()
    coverage = 1.0 - (null_count / df_sample.height)
    
    if coverage < 0.05: # Ignore if < 5% coverage
        continue
        
    # 2. Cardinality
    n_unique = df_sample[col].n_unique()
    
    # Ignore unique IDs (cardinality ~ rows) unless it's numeric range?
    if n_unique > df_sample.height * 0.95 and n_unique > 1000:
        continue
        
    # 3. Variance (if numeric)
    variance = 0.0
    is_numeric = dtype in (pl.Float64, pl.Float32, pl.Int64, pl.Int32)
    
    if is_numeric:
        variance = df_sample[col].var()
        if variance == 0 or np.isnan(variance):
            continue
    
    # 4. Value Counts (if categorical)
    if not is_numeric and n_unique == 1:
        continue
        
    # If passed filters, add to list
    stats.append({
        "column": col,
        "dtype": str(dtype),
        "coverage": coverage,
        "n_unique": n_unique,
        "is_numeric": is_numeric
    })

print(f"Candidate Features found: {len(stats)}")
df_stats = pl.DataFrame(stats).sort("coverage", descending=True)
print(df_stats.head(20))

# ## 2. Correlation Analysis (Top candidates)

In [ ]:
# Convert to pandas for easier correlation
valid_cols = [r["column"] for r in stats]
# Limit to top 50 by coverage/variance for display?
# Or just numeric ones for correlation matrix
numeric_candidates = [r["column"] for r in stats if r["is_numeric"]]
categorical_candidates = [r["column"] for r in stats if not r["is_numeric"]]

print(f"Numeric Candidates: {len(numeric_candidates)}")

# Calculate Correlation with Target
correlations = []
y = df_sample["is_fraud"].to_numpy()

for col in numeric_candidates:
    # Handle nulls for correlation (fill 0)
    x = df_sample[col].fill_null(0).to_numpy()
    
    # Simple correlation
    if x.std() == 0: continue
    
    corr = np.corrcoef(x, y)[0, 1]
    if not np.isnan(corr):
        correlations.append({"feature": col, "correlation": corr, "abs_corr": abs(corr)})

df_corr = pd.DataFrame(correlations).sort_values("abs_corr", ascending=False)
print("\nTop Numeric Correlations with Fraud:")
print(df_corr.head(20))

# ## 3. Categorical Information Signal (Mutual Information heuristic)

In [ ]:
# For high cardinality categoricals, correlation isn't great.
# Let's check fraud rate variance across categories.
# Ideally use Mutual Information, but simple "Max Fraud Rate Deviation" is a quick proxy.

cat_results = []
global_mean = y.mean()

for col in categorical_candidates:
    # Skip high cardinality > 1000 for this quick check to avoid noise
    # unless it's a hash we suspect (like city_hash)
    if "hash" not in col and df_sample[col].n_unique() > 100:
        continue
        
    # Group by category, calc fraud rate
    agg = (
        df_sample.group_by(col)
        .agg([
            pl.len().alias("count"),
            pl.col("is_fraud").mean().alias("fraud_rate")
        ])
        .filter(pl.col("count") > 50) # Min support
    )
    
    if agg.height < 2: continue
    
    # Score = Weighted deviation from mean?
    # Or just Max - Min fraud rate
    max_rate = agg["fraud_rate"].max()
    min_rate = agg["fraud_rate"].min()
    weighted_diff = (max_rate - min_rate) * (agg.height / 10) # rough
    
    cat_results.append({
        "feature": col,
        "max_fraud_rate": max_rate,
        "n_categories_screened": agg.height,
        "spread": max_rate - min_rate
    })

df_cat = pd.DataFrame(cat_results).sort_values("spread", ascending=False)
print("\nTop Categorical Features (Fraud Rate Spread):")
print(df_cat.head(20))

In [ ]:
# Save results to review
# In Jupyter we just display them
print("Top Stats:")
display(df_stats.head(20))
print("\nTop Correlations:")
display(df_corr.head(20))
print("\nTop Categorical:")
display(df_cat.head(20))